In [ ]:
import torch
import cv2
import ultralytics

print(f'torch {torch.__version__}, cuda {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

DATA_ROOT   = Path('../data/annotated')
SPLIT_DIR   = DATA_ROOT / 'split'
OUTPUT_DIR  = Path('../models/detector')
CONFIG_PATH = Path('../configs/detector_dataset.yaml')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
Path('../configs').mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np

images_dir = DATA_ROOT / 'images'
labels_dir = DATA_ROOT / 'labels'

image_files = sorted(images_dir.glob('*.jpg')) + sorted(images_dir.glob('*.png'))
label_files = sorted(labels_dir.glob('*.txt'))

counts = [len([l for l in lf.read_text().strip().splitlines() if l.strip()]) for lf in label_files]
print(f'images: {len(image_files)}, labels: {len(label_files)}, boxes total: {sum(counts)}')

In [ ]:
import shutil
import random
from sklearn.model_selection import train_test_split

SEED = 42

paired = [img for img in image_files if (labels_dir / (img.stem + '.txt')).exists()]

train_val, test = train_test_split(paired, test_size=0.05, random_state=SEED)
train, val      = train_test_split(train_val, test_size=0.15 / 0.95, random_state=SEED)

print(f'train {len(train)} / val {len(val)} / test {len(test)}')

def copy_split(files, split):
    for d in ['images', 'labels']:
        (SPLIT_DIR / split / d).mkdir(parents=True, exist_ok=True)
    for img in files:
        shutil.copy(img, SPLIT_DIR / split / 'images' / img.name)
        lbl = labels_dir / (img.stem + '.txt')
        shutil.copy(lbl, SPLIT_DIR / split / 'labels' / lbl.name)

copy_split(train, 'train')
copy_split(val,   'val')
copy_split(test,  'test')

In [ ]:
import yaml

CLASSES = ['price_tag']

cfg = {
    'path':  str(SPLIT_DIR.resolve()),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    len(CLASSES),
    'names': CLASSES,
}

CONFIG_PATH.write_text(yaml.dump(cfg, allow_unicode=True), encoding='utf-8')
print(yaml.dump(cfg, allow_unicode=True))

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

model.train(
    data    = str(CONFIG_PATH),
    epochs  = 100,
    imgsz   = 1280,
    batch   = 8,
    device  = 0,
    project = str(OUTPUT_DIR),
    name    = 'yolov8s_price_tags',
    exist_ok= True,

    lr0           = 0.01,
    lrf           = 0.01,
    warmup_epochs = 3,
    patience      = 20,

    degrees  = 10.0,
    translate= 0.1,
    scale    = 0.5,
    mosaic   = 1.0,
    mixup    = 0.0,
    flipud   = 0.0,
    fliplr   = 0.5,

    plots   = True,
    save    = True,
)

In [ ]:
best_weights = OUTPUT_DIR / 'yolov8s_price_tags' / 'weights' / 'best.pt'
model_best = YOLO(str(best_weights))

metrics = model_best.val(data=str(CONFIG_PATH), split='test')

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
import matplotlib.pyplot as plt

test_images = list((SPLIT_DIR / 'test' / 'images').glob('*.jpg'))
test_images += list((SPLIT_DIR / 'test' / 'images').glob('*.png'))

samples = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img_path in zip(axes.flatten(), samples):
    res = model_best.predict(str(img_path), conf=0.3, verbose=False)[0]
    ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f'{img_path.name}  [{len(res.boxes)}]')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
onnx_path = model_best.export(
    format   = 'onnx',
    opset    = 17,
    dynamic  = False,
    simplify = True,
)
print(f'saved: {onnx_path}')

import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
print('providers:', sess.get_providers())